# Covariance Matrices and Eigenanalysis
## Tutorial 3: From Raw Data to Signal/Noise Subspaces

The **spatial covariance matrix** is the central data structure for high-resolution DOA estimation.  This tutorial covers:

1. **Theoretical covariance** – structure from the signal model
2. **Sample covariance** – estimation from finite snapshots
3. **Eigendecomposition** – signal and noise eigenvalues/vectors
4. **Signal and noise subspaces** – the foundation of MUSIC, ESPRIT, …
5. **Effect of snapshots and SNR on subspace estimation**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

# Common setup
M = 8
array = UniformLinearArray(M=M, d=0.5)
signal_model = SignalModel(array)

doas_true = np.deg2rad([-20.0, 15.0])   # Two sources
K = len(doas_true)
snr_db = 10
N = 200                                   # Number of snapshots

print(f"Array: {M} elements, d=0.5λ")
print(f"Sources: {np.rad2deg(doas_true)} deg  |  SNR={snr_db} dB  |  N={N} snapshots")

## 1. Theoretical Covariance Matrix

For the signal model $\mathbf{x}(t) = \mathbf{A}\mathbf{s}(t) + \mathbf{n}(t)$, the **theoretical covariance matrix** is:

$$\mathbf{R} = E\{\mathbf{x}(t)\mathbf{x}^H(t)\} = \mathbf{A} \mathbf{P}_s \mathbf{A}^H + \sigma_n^2 \mathbf{I}$$

where $\mathbf{P}_s = \text{diag}(p_1, \ldots, p_K)$ is the source power matrix and $\sigma_n^2$ is the noise power.

This matrix is:
- **Hermitian**: $\mathbf{R} = \mathbf{R}^H$
- **Positive semi-definite**: $\mathbf{v}^H\mathbf{R}\mathbf{v} \geq 0$ for all $\mathbf{v}$
- **Rank-deficient** (due to sources): signal rank = $K$

In [ ]:
# Theoretical covariance
A = array.steering_vector(doas_true)       # M × K
snr_linear = 10**(snr_db/10)
P_s = snr_linear * np.eye(K)               # equal source powers
sigma_n2 = 1.0

R_theory = A @ P_s @ A.conj().T + sigma_n2 * np.eye(M)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(np.abs(R_theory), cmap='hot', aspect='auto')
axes[0].set_title('|R_theory|  (magnitude)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(np.angle(R_theory), cmap='bwr', aspect='auto',
                     vmin=-np.pi, vmax=np.pi)
axes[1].set_title('∠R_theory  (phase, rad)')
plt.colorbar(im1, ax=axes[1])

for ax in axes:
    ax.set_xlabel('Element j'); ax.set_ylabel('Element i')

plt.suptitle('Theoretical Covariance Matrix  (M=8, K=2, SNR=10 dB)', fontsize=13)
plt.tight_layout(); plt.show()

print("R_theory is Hermitian:", np.allclose(R_theory, R_theory.conj().T))
print("R_theory is PSD:", np.all(np.linalg.eigvalsh(R_theory) >= -1e-10))

## 2. Sample Covariance Matrix

In practice we only have $N$ snapshots, so we estimate:

$$\hat{\mathbf{R}} = \frac{1}{N} \sum_{t=1}^{N} \mathbf{x}(t)\mathbf{x}^H(t) = \frac{1}{N} \mathbf{X}\mathbf{X}^H$$

As $N \to \infty$, $\hat{\mathbf{R}} \to \mathbf{R}$ (law of large numbers).  For small $N$ the estimate is noisy.

In [ ]:
X, S, N_noise = signal_model.generate_signals(
    doas=doas_true, N_snapshots=N, snr_db=snr_db, seed=42)

R_sample = X @ X.conj().T / N

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
titles = ['Theoretical |R|', 'Sample |R̂|  (N=200)', 'Difference |R - R̂|']
mats = [np.abs(R_theory), np.abs(R_sample),
        np.abs(R_theory - R_sample)]

for ax, title, mat in zip(axes, titles, mats):
    im = ax.imshow(mat, cmap='hot', aspect='auto')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('j'); ax.set_ylabel('i')

plt.tight_layout(); plt.show()

error_frobenius = np.linalg.norm(R_theory - R_sample, 'fro')
print(f"Frobenius error |R - R̂|_F = {error_frobenius:.4f}")

## 3. Eigendecomposition

Any Hermitian matrix decomposes as:

$$\mathbf{R} = \mathbf{U} \boldsymbol{\Lambda} \mathbf{U}^H = \sum_{i=1}^{M} \lambda_i \mathbf{u}_i \mathbf{u}_i^H$$

where $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_M > 0$ and $\mathbf{U} = [\mathbf{u}_1, \ldots, \mathbf{u}_M]$ is unitary.

**Structure of eigenvalues for the signal model**:
- $K$ large eigenvalues: $\lambda_i \approx \sigma_n^2 + M p_i$  (signal + noise)
- $M-K$ small eigenvalues: $\lambda_i = \sigma_n^2$  (noise only)

The gap between the $K$-th and $(K+1)$-th eigenvalue is the **subspace signature**.

In [ ]:
eigenvals_th, eigenvecs_th = np.linalg.eigh(R_theory)
eigenvals_th = eigenvals_th[::-1]       # descending
eigenvecs_th = eigenvecs_th[:, ::-1]

eigenvals_sm, eigenvecs_sm = np.linalg.eigh(R_sample)
eigenvals_sm = eigenvals_sm[::-1]
eigenvecs_sm = eigenvecs_sm[:, ::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(range(1, M+1), eigenvals_th, 'bo-', ms=8, lw=2, label='Theoretical')
ax.semilogy(range(1, M+1), eigenvals_sm, 'rs--', ms=8, lw=2, label='Sample (N=200)')
ax.axvline(K + 0.5, color='gray', ls=':', lw=2, label=f'K={K} boundary')
ax.axhline(sigma_n2, color='orange', ls='--', lw=1.5, label=f'σ²_n = {sigma_n2}')
ax.set_xlabel('Eigenvalue index')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title('Eigenvalue Spectrum of Covariance Matrix')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Theoretical eigenvalues:", np.round(eigenvals_th, 4))
print("Noise floor (σ²_n):", sigma_n2)
print(f"Signal eigenvalues ({K} largest): {np.round(eigenvals_th[:K], 4)}")

## 4. Signal and Noise Subspaces

We partition the eigenvectors into:

$$\mathbf{U}_s = [\mathbf{u}_1, \ldots, \mathbf{u}_K] \quad (\text{signal subspace, } M\times K)$$
$$\mathbf{U}_n = [\mathbf{u}_{K+1}, \ldots, \mathbf{u}_M] \quad (\text{noise subspace, } M\times(M-K))$$

**Key property**: the signal subspace spans the same column space as $\mathbf{A}$:

$$\text{col}(\mathbf{U}_s) = \text{col}(\mathbf{A})$$

and therefore every steering vector $\mathbf{a}(\theta_k)$ is **orthogonal** to $\mathbf{U}_n$:

$$\mathbf{U}_n^H \mathbf{a}(\theta_k) = \mathbf{0}$$

This is the core insight exploited by MUSIC.

In [ ]:
# Extract subspaces from theoretical R
U_s = eigenvecs_th[:, :K]
U_n = eigenvecs_th[:, K:]

# Verify orthogonality: U_n^H a(θ_k) ≈ 0  for true DOAs
theta_grid = np.linspace(-np.pi/2, np.pi/2, 361)

projection_noise = np.zeros(len(theta_grid))
for i, th in enumerate(theta_grid):
    a = array.steering_vector(th).reshape(-1, 1)
    projection_noise[i] = np.real(a.conj().T @ U_n @ U_n.conj().T @ a).item()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(np.rad2deg(theta_grid), projection_noise, 'b-', lw=2, label='||U_n^H a(θ)||²')
for th_true in doas_true:
    ax.axvline(np.rad2deg(th_true), color='r', ls='--',
               label=f'True DOA {np.rad2deg(th_true):.0f}°')
ax.set_xlabel('θ (°)')
ax.set_ylabel('Projection onto noise subspace')
ax.set_title('Noise-Subspace Projection (theoretical R) — nulls at true DOAs')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Check orthogonality numerically
for k, th in enumerate(doas_true):
    a = array.steering_vector(th).reshape(-1, 1)
    proj = np.linalg.norm(U_n.conj().T @ a)
    print(f"||U_n^H a(θ={np.rad2deg(th):.0f}°)|| = {proj:.2e}  (should be ≈ 0)")

## 5. Effect of Snapshot Count on Subspace Quality

More snapshots → better covariance estimate → cleaner subspace separation.

In [ ]:
snapshot_counts = [10, 50, 200, 1000]
fig, axes = plt.subplots(1, len(snapshot_counts), figsize=(18, 5), sharey=True)

for ax, N_snap in zip(axes, snapshot_counts):
    X_, _, _ = signal_model.generate_signals(
        doas=doas_true, N_snapshots=N_snap, snr_db=snr_db, seed=0)
    R_hat = X_ @ X_.conj().T / N_snap
    evals, _ = np.linalg.eigh(R_hat)
    evals = evals[::-1]
    ax.semilogy(range(1, M+1), evals, 'ro-', ms=7, lw=1.8)
    ax.axvline(K + 0.5, color='gray', ls=':')
    ax.axhline(sigma_n2, color='orange', ls='--', lw=1.5)
    ax.set_title(f'N = {N_snap}')
    ax.set_xlabel('Index')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Eigenvalue')
plt.suptitle('Eigenvalue Spectrum vs Number of Snapshots', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Summary

- The theoretical covariance $\mathbf{R} = \mathbf{A}\mathbf{P}_s\mathbf{A}^H + \sigma_n^2\mathbf{I}$ has **rank-$K$ signal component** plus **scaled identity noise component**.
- Eigendecomposition cleanly separates the $K$ signal eigenvectors from the $(M-K)$ noise eigenvectors.
- The noise subspace is **orthogonal** to all true steering vectors — the basis of MUSIC.
- Sample covariance quality improves with $N$; very small $N$ ($N < M$) makes $\hat{\mathbf{R}}$ rank-deficient.

## Exercises
1. Show analytically that for equal source powers $p$ and noise power $\sigma_n^2$, the $K$ signal eigenvalues of $\mathbf{R}$ are all equal to $Mp + \sigma_n^2$ when the columns of $\mathbf{A}$ are orthogonal.
2. Use the sample covariance to compute the noise-subspace projection and compare the null locations with the theoretical result above.
3. What happens when $K$ is overestimated (you use $K'=3$ instead of $K=2$)?  Visualise the effect on the noise-subspace projection.